# TENSORIX v2: Knowledge-distillation checks, runs and results

Run cells **one at a time, top to bottom**. Do not use "Run all". Set the runtime to **T4 GPU** first.
Never run two GPU jobs at once. Rerun Cell 1 after every disconnect.
Cells 2 to 5 are one-time checks. Cell 6 trains one KD run. Cells 7 to 9 and the copy cell work with finished runs.

## Cell 1: Setup (run first in every session)

In [ ]:
import os, torch
BRANCH = "main"             # code lives on main after the PR is merged (use "feature/kd-v2" only before the merge)
REPO   = "https://github.com/sunidhishekar-create/tensorix-kd-slm.git"
DRIVE  = "/content/drive/MyDrive/tensorix/v2"
TEACHER = "HuggingFaceTB/SmolLM2-360M"

from google.colab import drive
drive.mount("/content/drive")

%cd /content
if not os.path.exists("/content/tensorix-kd-slm"):
    !git clone {REPO}
%cd /content/tensorix-kd-slm
!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
!git log --oneline -5
!pip install -q -r requirements.txt

os.makedirs("/content/data", exist_ok=True)
if not os.path.exists("/content/data/train.bin"):
    !cp {DRIVE}/data/* /content/data/
!ls -lh /content/data
need = ["train.bin", "val_c4.bin", "val_ts.bin", "meta.json"]
missing = [f for f in need if not os.path.exists(f"/content/data/{f}")]
assert not missing, f"missing in /content/data: {missing}"
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then rerun Cell 1."
print("GPU:", torch.cuda.get_device_name(0), "| setup OK")

## Cell 2: Unit tests of the KD loss (about 1 minute). Expected: every line OK and a final all-passed line.

In [ ]:
%cd /content/tensorix-kd-slm
!python src/distill.py --selftest

## Cell 3: Does the teacher loader reproduce the README numbers?
Expected: SmolLM2-360M scores C4 loss about **3.047** and TinyStories about **1.907** (within about 0.01).

In [ ]:
%cd /content/tensorix-kd-slm
!python src/distill.py --check_teacher --data_dir /content/data --teacher {TEACHER}

## Cell 4: fp16 teacher versus fp32 teacher. Expected: no NaN or inf, small KL, high top-1 agreement.

In [ ]:
%cd /content/tensorix-kd-slm
!python src/distill.py --check_fp16 --data_dir /content/data --teacher {TEACHER}

## Cell 5: Speed and memory test (100 steps each)
Measured on a T4: baseline about 514 to 556 ms/step; KD about 2,000 to 2,300 ms/step; peak memory about 11.2 GiB.
If you get CUDA out of memory, add `"--grad_accum", "4"` to the KD command.

In [ ]:
import subprocess, sys
def run(cmd):
    """Run a command and stream its output into the notebook."""
    env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print(f"[exit code {p.returncode}]")

%cd /content/tensorix-kd-slm
!rm -rf /content/speedtest_base /content/speedtest_kd
common = ["--data_dir", "/content/data", "--max_steps", "100", "--eval_interval", "100", "--save_interval", "100"]
print("=== BASELINE, 100 steps ===")
run([sys.executable, "src/train.py", "--out_dir", "/content/speedtest_base"] + common)
print("=== KD (T=2, alpha=0.5), 100 steps ===")
run([sys.executable, "src/train.py", "--out_dir", "/content/speedtest_kd"] + common +
    ["--teacher", TEACHER, "--kd_alpha", "0.5", "--kd_temp", "2"])

## Cell 6: One real KD run, SHORT tier (5,000 steps, about 3.1 to 3.4 hours on a T4)
Set `KD_T` and `RUN_NAME` (naming rule: `kd_short_T<temperature>`). Alpha stays 0.5 and the steps stay 5,000 for the whole sweep.
The run trains on Colab's local disk and syncs to Drive every 5 minutes. After a disconnect, rerun Cell 1 and then this cell UNCHANGED (same `RUN_NAME`): it resumes from the last saved copy.
If syncing into the shared folder fails, set `SYNC_ROOT` to a folder in your own Drive (for example `/content/drive/MyDrive/kd_runs`) and move the finished run afterwards.

In [ ]:
import os, sys, subprocess, threading, shutil

KD_T      = 2
KD_ALPHA  = 0.5
RUN_NAME  = "kd_short_T2"
MAX_STEPS = 5000
SYNC_ROOT = f"{DRIVE}/checkpoints"        # or "/content/drive/MyDrive/kd_runs" if the shared folder rejects writes

def run(cmd):
    env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout: print(line, end="")
    p.wait(); print(f"[exit code {p.returncode}]")

LOCAL     = f"/content/runs/{RUN_NAME}"
DRIVE_OUT = f"{SYNC_ROOT}/{RUN_NAME}"
FILES     = ("log.csv", "best.pt", "ckpt.pt", "run_args.json")
os.makedirs(LOCAL, exist_ok=True)
if os.path.exists(f"{DRIVE_OUT}/ckpt.pt"):                 # resume after a disconnect
    shutil.copytree(DRIVE_OUT, LOCAL, dirs_exist_ok=True)
    print("Resuming from the copy in", DRIVE_OUT)

def push():
    try:
        os.makedirs(DRIVE_OUT, exist_ok=True)
        for f in FILES:
            s, d = f"{LOCAL}/{f}", f"{DRIVE_OUT}/{f}"
            if os.path.exists(s) and (not os.path.exists(d) or os.path.getmtime(s) > os.path.getmtime(d)):
                shutil.copy2(s, d + ".tmp"); os.replace(d + ".tmp", d)
    except Exception as e:
        print("!! sync to Drive failed:", e)

stop = threading.Event()
threading.Thread(target=lambda: [push() for _ in iter(lambda: stop.wait(300), True)], daemon=True).start()

%cd /content/tensorix-kd-slm
try:
    run([sys.executable, "src/train.py", "--data_dir", "/content/data", "--out_dir", LOCAL,
         "--max_steps", str(MAX_STEPS), "--eval_interval", "250", "--save_interval", "500",
         "--teacher", TEACHER, "--kd_alpha", str(KD_ALPHA), "--kd_temp", str(KD_T)])
finally:
    stop.set(); push()
print("In Drive:", os.listdir(DRIVE_OUT) if os.path.isdir(DRIVE_OUT) else "SYNC FAILED")

## Cell 7: Best-step check and log copies (no GPU needed)
Checks that every run's best validation step is the final step (5000), so the checkpoint scored later is the final model. It also copies the logs for `results/kd/` and downloads them as a zip.

In [ ]:
import os, shutil, pandas as pd

DRIVE = "/content/drive/MyDrive/tensorix/v2"
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

RUNS   = ["baseline_short", "kd_short_T2", "kd_short_T4"]
TARGET = 5000
OUT    = "/content/results_kd"
os.makedirs(OUT, exist_ok=True)

def ckpt_dir(run):
    for root in (f"{DRIVE}/checkpoints", "/content/drive/MyDrive/kd_runs", "/content/runs"):
        if os.path.isdir(f"{root}/{run}"):
            return f"{root}/{run}"
    return None

summary, lines, all_pass = [], [], True
for r in RUNS:
    d = ckpt_dir(r)
    if not d or not os.path.exists(f"{d}/log.csv"):
        print(f"[MISSING] {r}: no log.csv found"); all_pass = False; continue
    raw = pd.read_csv(f"{d}/log.csv")
    dup = int(raw.duplicated("step").sum())
    df = raw.drop_duplicates("step", keep="last").sort_values("step").reset_index(drop=True)
    steps = df.step.astype(int)
    last     = int(steps.iloc[-1])
    interval = int(steps.diff().dropna().mode().iloc[0]) if len(df) > 1 else None
    best_all = int(df.loc[df.val_avg.idxmin(), "step"])
    d500     = df[steps % 500 == 0]
    best_500 = int(d500.loc[d500.val_avg.idxmin(), "step"]) if len(d500) else None
    ok = (last == TARGET) and (best_all == last) and (best_500 == best_all)
    all_pass &= ok
    summary.append(dict(run=r, rows=len(df), interval=interval, last=last, best_all=best_all,
                        best_500s=best_500, dup_rows=dup, verdict="PASS" if ok else "CHECK"))
    lines.append(f"Run '{r}': Best Step (All) = {best_all} | Best Step (500s) = {best_500} | Last Step = {last}")
    shutil.copy2(f"{d}/log.csv", f"{OUT}/{r}_log.csv")

print(pd.DataFrame(summary).to_string(index=False))
print("\nLines for the README:")
for l in lines: print("  ", l)
print("\nOverall:", "PASS" if all_pass else "CHECK: paste this output to the reviewer/teammate before writing 'unaffected'")
open(f"{OUT}/best_step_check.txt", "w").write("\n".join(lines) + "\n")
shutil.make_archive("/content/results_kd", "zip", OUT)
try:
    from google.colab import files
    files.download("/content/results_kd.zip")
except Exception:
    print("Download /content/results_kd.zip from the Colab file panel.")

## Cell 8: Table and curves (no GPU needed)

In [ ]:
import os, pandas as pd, matplotlib.pyplot as plt

DRIVE = "/content/drive/MyDrive/tensorix/v2"
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

RUNS       = ["baseline_short", "kd_short_T2", "kd_short_T4"]
BASELINE   = "baseline_short"
TEACHER_C4 = 3.047

def ckpt_dir(run):
    for root in (f"{DRIVE}/checkpoints", "/content/drive/MyDrive/kd_runs", "/content/runs"):
        if os.path.isdir(f"{root}/{run}"):
            return f"{root}/{run}"
    return None

logs = {}
for r in RUNS:
    d = ckpt_dir(r)
    if not d or not os.path.exists(f"{d}/log.csv"):
        print("missing run:", r); continue
    df = pd.read_csv(f"{d}/log.csv").drop_duplicates("step", keep="last").sort_values("step")
    logs[r] = df
    print(f"{r}  ({d})"); print(df.tail(3).to_string(index=False)); print()

if BASELINE in logs:
    b = logs[BASELINE].iloc[-1]
    print(f"{'run':16s} {'step':>5s} {'C4 loss':>8s} {'C4 ppl':>8s} {'TS loss':>8s} {'TS ppl':>8s} {'dC4 ppl':>8s} {'dTS ppl':>8s} {'gap closed C4':>14s}")
    for r, df in logs.items():
        k = df.iloc[-1]
        d_c4 = 100 * (k.ppl_c4 / b.ppl_c4 - 1)
        d_ts = 100 * (k.ppl_ts / b.ppl_ts - 1)
        gap = (b.val_c4 - k.val_c4) / (b.val_c4 - TEACHER_C4)
        flag = "" if int(k.step) == int(b.step) else "   <-- last step differs from baseline"
        print(f"{r:16s} {int(k.step):5d} {k.val_c4:8.4f} {k.ppl_c4:8.1f} {k.val_ts:8.4f} {k.ppl_ts:8.2f} {d_c4:+7.1f}% {d_ts:+7.1f}% {gap:14.3f}{flag}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for r, df in logs.items():
    ax[0].plot(df.step, df.ppl_c4, marker="o", markersize=3, label=r)
    ax[1].plot(df.step, df.ppl_ts, marker="o", markersize=3, label=r)
for a, name in zip(ax, ["C4", "TinyStories"]):
    a.set(title=f"Validation perplexity: {name}", xlabel="step", ylabel="perplexity", yscale="log")
    a.grid(alpha=0.3); a.legend()
plt.savefig("/content/val_ppl_kd.png", dpi=150, bbox_inches="tight")
try:
    os.makedirs(f"{DRIVE}/results", exist_ok=True)
    plt.savefig(f"{DRIVE}/results/val_ppl_kd.png", dpi=150, bbox_inches="tight")
    print("plot also saved to Drive")
except Exception as e:
    print("could not save to Drive (local copy /content/val_ppl_kd.png exists):", e)
plt.show()

## Cell 9: Score finished runs with evaluate.py (needs a GPU; run when no training is using it)
The losses printed here must match the last row of each run's `log.csv` to about 3 decimals.

In [ ]:
import os, sys, json, subprocess, pandas as pd, torch

DRIVE    = "/content/drive/MyDrive/tensorix/v2"
EVAL_DIR = f"{DRIVE}/results/eval"
EVAL_KD  = ["kd_short_T2", "kd_short_T4"]
TEACHER_C4 = 3.047

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then rerun Cell 1."
assert os.path.exists("/content/tensorix-kd-slm/src/evaluate.py"), "Run Cell 1 first (repo missing)."
assert os.path.exists("/content/data/val_c4.bin"), "Run Cell 1 first (data missing)."

def ckpt_dir(run):
    for root in (f"{DRIVE}/checkpoints", "/content/drive/MyDrive/kd_runs", "/content/runs"):
        if os.path.isdir(f"{root}/{run}"):
            return f"{root}/{run}"
    return None

def run(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: print(line, end="")
    p.wait(); print(f"[exit code {p.returncode}]")

os.makedirs(EVAL_DIR, exist_ok=True)
os.chdir("/content/tensorix-kd-slm")
for name in EVAL_KD:
    d = ckpt_dir(name)
    if not d or not os.path.exists(f"{d}/best.pt"):
        print("no best.pt for", name, "- skipping"); continue
    print("=== scoring", name, "from", d, "===")
    run([sys.executable, "src/evaluate.py", "--data_dir", "/content/data",
         "--ckpt", f"{d}/best.pt", "--name", name, "--out_dir", EVAL_DIR])

def L(name, key):
    return json.load(open(f"{EVAL_DIR}/{name}.json"))[key]

try:
    b = L("baseline_short", "val_c4_loss")
    try:
        t = L("teacher_base", "val_c4_loss")
    except FileNotFoundError:
        t = TEACHER_C4
        print(f"(teacher_base.json missing; using README teacher C4 loss {t})")
    print(f"\nC4 loss: baseline_short {b:.4f} | teacher {t:.4f}")
    for name in EVAL_KD:
        if not os.path.exists(f"{EVAL_DIR}/{name}.json"): continue
        c4, ts = L(name, "val_c4_loss"), L(name, "val_ts_loss")
        print(f"{name}: C4 loss {c4:.4f} | TS loss {ts:.4f} | gap closed on C4 = {(b - c4) / (b - t):.3f}")
        last = pd.read_csv(f"{ckpt_dir(name)}/log.csv").drop_duplicates("step", keep="last").iloc[-1]
        diff = max(abs(c4 - last.val_c4), abs(ts - last.val_ts))
        print(f"   check vs log.csv last row (step {int(last.step)}): max difference {diff:.4f}",
              "OK" if diff < 0.005 else "<-- MISMATCH")
except FileNotFoundError as e:
    print("missing JSON:", e)

## Copy cell: move a finished run into the shared checkpoints folder
Only after the run printed `done`. It copies, verifies sizes and checksums, rechecks after 30 s, and only deletes the original if you set `DELETE_SOURCE = True`.

In [ ]:
import os, time, shutil, hashlib, pandas as pd

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

RUN   = "kd_short_T4"
SRC   = f"/content/drive/MyDrive/kd_runs/{RUN}"
DST   = f"/content/drive/MyDrive/tensorix/v2/checkpoints/{RUN}"
FILES = ("log.csv", "best.pt", "ckpt.pt", "run_args.json")
EXPECTED_LAST_STEP = 5000
ALLOW_OVERWRITE = False
DELETE_SOURCE   = False

def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

assert os.path.isdir(SRC), f"source folder not found: {SRC}"
missing = [f for f in FILES if not os.path.exists(f"{SRC}/{f}")]
assert not missing, f"missing in source: {missing}"
last = int(pd.read_csv(f"{SRC}/log.csv").step.max())
assert last == EXPECTED_LAST_STEP, f"run is not finished: last logged step is {last}"
if os.path.isdir(DST) and os.listdir(DST) and not ALLOW_OVERWRITE:
    raise SystemExit(f"Stopped: destination already has files {os.listdir(DST)}. Set ALLOW_OVERWRITE = True if sure.")

os.makedirs(DST, exist_ok=True)
for f in FILES:
    print("copying", f, "...")
    shutil.copy2(f"{SRC}/{f}", f"{DST}/{f}.tmp")
    os.replace(f"{DST}/{f}.tmp", f"{DST}/{f}")

def verify():
    ok = True
    for f in FILES:
        s, d = f"{SRC}/{f}", f"{DST}/{f}"
        if not os.path.exists(d):
            print("  MISSING in destination:", f); ok = False; continue
        same_size = os.path.getsize(s) == os.path.getsize(d)
        same_hash = True if f.endswith(".pt") else (md5(s) == md5(d))
        print(f"  {f:14s} size match: {same_size} | checksum match: {same_hash if not f.endswith('.pt') else 'skipped'}")
        ok &= same_size and same_hash
    return ok

ok1 = verify()
print("waiting 30 s and checking again ...")
time.sleep(30)
ok2 = verify() and os.path.isdir(DST)
print("\nRESULT:", "COPY VERIFIED" if (ok1 and ok2) else "PROBLEM - keep the original, do not delete it")
if ok1 and ok2 and DELETE_SOURCE:
    shutil.rmtree(SRC); print("original removed:", SRC)